# XLM-R experiments (V2) — run on Colab GPU

Runtime → Change runtime type → **T4 GPU**.

Upload `smishing_v2_colab.zip` to your Google Drive (`MyDrive/smishing/`). It contains `src/` and `data/v2/` (private — do not share the Drive folder). The 163 real SMS are used **only** for the final evaluation; validation for early stopping / thresholds is a family-grouped split of the training data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/sd && cd /content/sd && unzip -oq /content/drive/MyDrive/smishing/smishing_v2_colab.zip
%cd /content/sd
!pip -q install transformers==4.44.2 sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Token length analysis (choose max_len)

In [ ]:
!python -m src.v2.xlmr_train --token-stats --max_len 128

## 2. Binary: learning-rate × batch-size sweep (selection on the validation split only)

In [ ]:
for lr in ['1e-5', '2e-5', '3e-5']:
    for bs in ['16', '32']:
        !python -m src.v2.xlmr_train --task binary --lr {lr} --bs {bs} --epochs 6 --patience 2

In [ ]:
import pandas as pd, glob, json
rows=[]
for d in glob.glob('results/v2/xlmr/binary_*'):
    m=pd.read_csv(d+'/metrics.csv'); v=m[m.set=='validation'].iloc[0]
    rows.append({'run':d.split('/')[-1],'val_auprc':v.auprc,'val_f1':v.f1})
sweep=pd.DataFrame(rows).sort_values('val_auprc',ascending=False); sweep

## 3. Three-class with the best binary learning rate / batch size

In [ ]:
best=sweep.iloc[0].run; lr=best.split('_lr')[1].split('_')[0]; bs=best.split('_bs')[1].split('_')[0]
!python -m src.v2.xlmr_train --task three --lr {lr} --bs {bs} --epochs 6 --patience 2

## 4. Seed robustness of the selected binary configuration

In [ ]:
for seed in ['7', '2026']:
    !python -m src.v2.xlmr_train --task binary --lr {lr} --bs {bs} --seed {seed}

## 5. Save results (metrics, histories, predictions) back to Drive
Checkpoints are large; copy them only if you need them for the API.

In [ ]:
!zip -qr /content/drive/MyDrive/smishing/xlmr_results.zip results/v2/xlmr
# optional: !zip -qr /content/drive/MyDrive/smishing/xlmr_models.zip models/v2/xlmr
print('saved')